In [432]:
#
# invoke via:
# jupyter notebook AWS-icons-scraping-selenium.ipynb
#
# then run each cell
#

from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import sys
import time
#
# need to:
#
# install selenium
# install chrome driver
# xattr -d com.apple.quarantine chromedriver
# chromedriver is in the top level folder, chromewebdriver -->



cService = webdriver.ChromeService(executable_path="/Users/william/chromewebdriver/chromedriver-mac-arm64/chromedriver")

driver = webdriver.Chrome(service=cService)

#driver.get('https://aws-icons.com/category/analytics') # for testing purposes
driver.get('https://aws-icons.com/') # 311 of 'em

# may need to adjust the timeout based upon experience...
wait = WebDriverWait(driver, 10)
new_count = 0
old_count = 0
# this code loads the entire web page. At the time of this scraping, the page was dynamically loaded as you scrolled
while True:
    old_count = new_count

    services = wait.until(EC.visibility_of_all_elements_located((By.XPATH, './/div[@class="v-popper v-popper--theme-tooltip"]')))
    anchors  = wait.until(EC.visibility_of_all_elements_located((By.XPATH, './/div[@class="v-popper v-popper--theme-tooltip"]//a')))

    new_count = len(services)
    
    # scroll down to last service to fully load the page
    # this invokes Javascript code
    driver.execute_script("arguments[0].scrollIntoView();", services[len(services) - 1])

    # if the count didn't change, we've loaded all services on the page
    # I put a max of 311 services to load as a demo. You can adjust higher as needed but you should put something reasonably sized here to prevent the script from running for an hour
    # total number of services
    if new_count == old_count or new_count > 311:
        break

n = len(services) 
# first one is bogus so we skip over it
services = services[1:n]

# print results

print('# of services: ',len(services) )
print('# of anchors: ', len(anchors))

#
# iterate over services and get hrefs
#

services_list = []
href_list = []

i = 0

for service in services:
     # we need to wait for the DOM

    #anchorElement = wait.until(EC.element_located_to_be_selected((By.XPATH, './/a')))
    anchorElement = service.find_element(By.XPATH, './/a')

    #imgElement = service.find_element(By.XPATH, './/a//div//div//img')
    #imgElement = service.find_element(By.XPATH, './/a//div//div//img')
    #imgElement = wait.until(EC.element_to_be_selected((By.XPATH, './/a//div//div//img')))
    services_list.append(service.text)
    href_list.append(anchorElement.get_attribute('href'))

    print(i,services_list[i], href_list[i])
# test code:
#    if i > 20:
#        break
    i += 1


# of services:  311
# of anchors:  311
0 Athena https://aws-icons.com/icons/athena
1 Clean Rooms https://aws-icons.com/icons/clean-rooms
2 CloudSearch https://aws-icons.com/icons/cloudsearch
3 Data Exchange https://aws-icons.com/icons/data-exchange
4 Data Pipeline https://aws-icons.com/icons/data-pipeline
5 DataZone https://aws-icons.com/icons/datazone
6 EMR https://aws-icons.com/icons/emr
7 FinSpace https://aws-icons.com/icons/finspace
8 Glue https://aws-icons.com/icons/glue
9 Glue DataBrew https://aws-icons.com/icons/glue-databrew
10 Glue Elastic Views https://aws-icons.com/icons/glue-elastic-views
11 Kinesis https://aws-icons.com/icons/kinesis
12 Kinesis Data Analytics https://aws-icons.com/icons/kinesis-data-analytics
13 Kinesis Data Streams https://aws-icons.com/icons/kinesis-data-streams
14 Kinesis Firehose https://aws-icons.com/icons/kinesis-firehose
15 Kinesis Video Streams https://aws-icons.com/icons/kinesis-video-streams
16 Lake Formation https://aws-icons.com/icons/lake-form

In [434]:
#
# get descriptions and image attributes
#
descriptions = []
images_list = []

test_links= [ 'https://aws-icons.com/icons/athena',
'https://aws-icons.com/icons/clean-rooms',
'https://aws-icons.com/icons/cloudsearch' ]    
# go to the link; assumes href_list is populated in the cell above
for i in range(len(href_list)):
    
    # grab the page
    
    driver.get(href_list[i])
    
    # get the description from the first 'p' element
    
    driver.implicitly_wait(2) # we need to wait for the DOM

    descriptions.append( driver.find_element(By.XPATH, './/p[@class="md:text-lg text-zinc-500"]').text.strip() )
    #imgElement = 
    images_list.append(driver.find_element(By.XPATH, '//html/body/div/div/div/div/div/div/div/div/div/img').get_attribute('src') )
    images_list[i] = images_list[i].replace('https://icon.icepanel.io/AWS/svg/','imgs/') 
    
    # create reference html link for when clicking on icon
    
    href_list[i] = href_list[i].replace('https://aws-icons.com/icons/', 'https://aws.amazon.com/')
    print(href_list[i], images_list[i])
#print(descriptions)
driver.close()


https://aws.amazon.com/athena imgs/Analytics/Athena.svg
https://aws.amazon.com/clean-rooms imgs/Analytics/Clean-Rooms.svg
https://aws.amazon.com/cloudsearch imgs/Analytics/CloudSearch.svg
https://aws.amazon.com/data-exchange imgs/Analytics/Data-Exchange.svg
https://aws.amazon.com/data-pipeline imgs/Analytics/Data-Pipeline.svg
https://aws.amazon.com/datazone imgs/Analytics/DataZone.svg
https://aws.amazon.com/emr imgs/Analytics/EMR.svg
https://aws.amazon.com/finspace imgs/Analytics/FinSpace.svg
https://aws.amazon.com/glue imgs/Analytics/Glue.svg
https://aws.amazon.com/glue-databrew imgs/Analytics/Glue-DataBrew.svg
https://aws.amazon.com/glue-elastic-views imgs/Analytics/Glue-Elastic-Views.svg
https://aws.amazon.com/kinesis imgs/Analytics/Kinesis.svg
https://aws.amazon.com/kinesis-data-analytics imgs/Analytics/Kinesis-Data-Analytics.svg
https://aws.amazon.com/kinesis-data-streams imgs/Analytics/Kinesis-Data-Streams.svg
https://aws.amazon.com/kinesis-firehose imgs/Analytics/Kinesis-Firehos

In [436]:
#
# after running this cell the data structure now looks like:
#
# [
#  {
#   'id': 0,
#   'name': 'Athena',
#   'image': 'imgs/Analytics/Athena.svg',
#   'url': 'https://aws-icons.com/icons/athena',
#   'description': 'Amazon Athena is a serverless, interactive analytics service that provides ...'
#  },
#  {
#   'id':1,
#   'name': 'Clean Rooms',
#   'image': 'imgs/Analytics/Clean-Rooms.svg', 
#   'url': 'https://aws-icons.com/icons/clean-rooms',
#   'description': 'AWS Clean Rooms helps companies and their partners more securely analyze and collaborate on the...'
#  }
# ]
# so we have to parse an array of objects on input into our data visualization javascript code.
#                                                                                                                                                                                                                                                                                        'url': 'https://aws-icons.com/icons/cloudsearch', 
# remove remove all spaces, tabs and newlines and then put the string back together again. There is an indentation error thrown
# and this corrects it.
#'description': 'Quickly add rich, scalable search capabilities to your website or application with support for 34 languages and search features like highlighting, autocomplete, and geospatial search.'}
#
descriptions_clean = []
for i in range(len(descriptions)):
    descriptions_clean.append(' '.join(descriptions[i].split()))
    print(i, descriptions_clean[i])
# from the docs re: split():
# If sep is not specified or is None, a different splitting algorithm is applied: 
# runs of consecutive whitespace are regarded as a single separator, and the 
# result will contain no empty strings at the start or end if the string has leading or trailing whitespace.

#output = [{"name": services_list[i], "image": images_list[i], "url": href_list[i], "description": descriptions[i]} for i in range(len(services))]
output = [{"id": i,"name": services_list[i], "image": images_list[i], "url": href_list[i], "description": descriptions_clean[i]} for i in range(len(services_list))]
print(output)

0 Amazon Athena is a serverless, interactive analytics service that provides a simplified and flexible way to analyze petabytes of data where it lives.
1 AWS Clean Rooms helps companies and their partners more securely analyze and collaborate on their collective datasets without sharing or revealing underlying data.
2 Quickly add rich, scalable search capabilities to your website or application with support for 34 languages and search features like highlighting, autocomplete, and geospatial search.
3 There is no other place where customers can find data files, data tables, and data APIs from a vast portfolio of third-party data sets. We continuously innovate to make the world's third-party data easy to find in one data catalog.
4 AWS Data Pipeline is a cloud-based data workflow service that helps you process and move data between different AWS services and on-premise data sources.
5 Use Amazon DataZone to discover, share, govern, and analyze data at scale across organizational boundari

In [438]:
#
# this is the final cell to run to write out to a json file
#
import json

# Output:
# This will create a file named 'services.json' and write the JSON data into it.
with open('services.json', 'w') as f:
    json.dump(output, f)

f.close()

In [ ]:
#
# this code was the initial foray into scraping the aws-icons.com website.
#
# I learned that the page was not loading fully so my html element XPath queries were failing in random places,
# working in test mode (less number of elements loaded) and then I realized that the page wasn't fully loaded. 
# the code in the first cell shows the solution to this problem. This is just legacy code.
# invoke via:
# jupyter notebook AWS-icons-scraping-selenium.ipynb
#
# then run each cell
#

from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import sys
import time
import csv
#
# need to:
#
# install selenium
# install chrome driver
# xattr -d com.apple.quarantine chromedriver
# chromedriver is in the top level folder, chromewebdriver -->

cService = webdriver.ChromeService(executable_path="/Users/william/chromewebdriver/chromedriver-mac-arm64/chromedriver")

driver = webdriver.Chrome(service=cService)

#driver.get('https://aws-icons.com/category/analytics') # for testing purposes
driver.get('https://aws-icons.com/') # 311 of 'em

driver.implicitly_wait(.5) # only need to call this once according to the docs
div = driver.find_element(By.XPATH, './/div[@class="carousel__viewport"]')
time.sleep(1)
categories = div.find_elements(By.XPATH, './/ol//li//div//a')

#
# get all 311 services
#


#n = len(services) 
# first one is bogus so we skip over it
#services = services[1:n]
#print(services[310].text)
print('It seem everything is okay.',len(categories), 'services found.')



# Wait until the page is fully loaded
try:
    # Waiting for the presence of an element on the page
    element_present = EC.presence_of_element_located((By.ID, 'element_id'))
    WebDriverWait(driver, 5).until(element_present) # 5 = five seconds
    print("Page is ready!")
except TimeoutException:
    print("Loading took too much time!")

# Continue with your scraping tasks here

# Close the WebDriver
#driver.close()
#driver.quit() 



In [ ]:
#
# this was my batch/do-it-by-chunk approach but I still needed to load the entire page to parse it properly.
#
categories_list = []
for i in range(len(categories)):
    categories_list.append(categories[i].get_attribute('href'))
print(categories_list)

# copy and paste output from above the output of above as we want to control which services we go after to get around the problem of 
# one fell swoop doesn't seem to work for one category so we are going to isolate that category.

batch1 = ['https://aws-icons.com/category/analytics']
          
batch2 = ['https://aws-icons.com/category/app-integration',
          'https://aws-icons.com/category/blockchain',
          'https://aws-icons.com/category/business-applications',
          'https://aws-icons.com/category/cloud-financial-management']
batch3 = ['https://aws-icons.com/category/compute']
batch4 = ['https://aws-icons.com/category/containers',
          'https://aws-icons.com/category/customer-enablement',
          'https://aws-icons.com/category/database']
batch5 = ['https://aws-icons.com/category/developer-tools',
          'https://aws-icons.com/category/end-user-computing',
          'https://aws-icons.com/category/front-end-web-mobile',
          'https://aws-icons.com/category/games',
          'https://aws-icons.com/category/general-icons']
batch6 = ['https://aws-icons.com/category/internet-of-things']
batch7 = ['https://aws-icons.com/category/machine-learning']
batch8 = ['https://aws-icons.com/category/management-governance',
          'https://aws-icons.com/category/media-services']
batch9 = ['https://aws-icons.com/category/migration-transfer',
          'https://aws-icons.com/category/networking-content-delivery',
          'https://aws-icons.com/category/quantum-technologies',
          'https://aws-icons.com/category/robotics',
          'https://aws-icons.com/category/satellite',
          'https://aws-icons.com/category/security-identity-compliance',
          'https://aws-icons.com/category/storage',
          'https://aws-icons.com/category/vr-ar']

In [ ]:
#
# Yep, more legacy code. Preserved for prosperity's sake.
#
# iterate over services and get hrefs and images
#

services_list = []
href_list = []
images_list = []
# iterate over each category
categories_list = batch7
for j in range(len(categories_list)):
    print(j, categories_list[j])
    driver.get(categories_list[j])
    
    driver.implicitly_wait(2) # we need to wait for the DOM

    time.sleep(1)
# get services for this category
    services = driver.find_elements(By.XPATH,'.//div[@class="v-popper v-popper--theme-tooltip"]')
 
        
    n = len(services) 
    # first one is bogus so we skip over it
    services = services[1:n]
    # iterate over each service and get the href attribute, like 'icons/athena'
    print('# of services: ',len(services) )
    for i in range(len(services)):
        driver.implicitly_wait(2) # we need to wait for the DOM
        time.sleep(1)
        # create the name field
        services_list.append(services[i].text)

        driver.implicitly_wait(2) # we need to wait for the DOM

        # get the link to go get the desciption for the tooltip
        href_list.append(services[i].find_element(By.XPATH, './/a').get_attribute('href'))
        time.sleep(1)
        print(i, services_list[i], href_list[i])

        driver.implicitly_wait(2) # we need to wait for the DOM

        # get the images
        # you have to tell selenium where to descend down the DOM - from here (service) down an anchor, through two divs, to the img.
        # based upon inspection of the element
        #images_list.append(services[i].find_element(By.XPATH,'//*[@id="analytics"]/div[2]/div[2]/div[1]/a/div/div/img').get_attribute('src').replace('https://icon.icepanel.io/AWS/svg/','imgs/') )
        imgElement = services[i].find_element(By.XPATH, './/a//div//div//img')
        driver.implicitly_wait(2) # we need to wait for the DOM
        time.sleep(1)

        images_list.append(imgElement.get_attribute('src').replace('https://icon.icepanel.io/AWS/svg/','imgs/') )

    services = [] # maybe not needed
# turn this off in prod
print(services_list, href_list, images_list)


In [ ]:
#
# this is how you merge lists together in anticipation of outputting to a csv file. 
# CSV files are easier to append to as opposed to JSON. Remember I was in chunk mode.
#

rows = zip(services_list, images_list, href_list, descriptions)

#
# output to an EXISTING csv file
#


import csv
file = open('services.csv', 'a', newline='')
writer = csv.writer(file)
for row in rows:
    print(row)
    writer.writerow(row)

file.close()
    